# Debt Capacity & Credit Structuring Analysis
**Portfolio Project | Leveraged Finance / Acquisition Financing**

Author: Brian Cotton

Date: November 2026  
Background: 15 years risk management experience applied to IB-style credit analysis

## Executive Summary
- Modeled maximum sustainable leverage for a USD 80mm LTM EBITDA industrial target under base and downside cases.
- Base case supports ~7.17x leverage (interest coverage 2.0x); downside ~7.78x (1.5x coverage).
- Year-1 DSCR is tight (~1.09x base / 0.80x downside) with 5% amortization.
- Proposed senior secured structure: USD 600mm term loan + USD 25mm revolver, with specific covenants and ECF sweep.
- Primary credit flags: limited coverage/DSCR headroom at maximum leverage.

### 1. Problem Statement
#### You are advising on financing an acquisition of a stable industrial business. The target has:

* LTM EBITDA: $80
* Projected Year 1 EBITDA: \\$86 (base), with a downside case of \$70 (15% haircut)
* Maintenance Capex: \$8
* Cash taxes roughly 25% of EBIT (assume EBIT ≈ EBITDA – \$10 D&A)
* No material working capital swings in the base case
* Desired structure: senior secured term loan + optional revolver

#### Tasks:

* Estimate maximum total leverage (Debt/EBITDA) that keeps Year 1 interest coverage (EBITDA / cash interest) ≥ 2.0x in the base case and ≥ 1.5x in the downside case. Assume an all-in cash interest rate of 7.5% on senior debt.
* Calculate approximate Debt Service Coverage Ratio (DSCR = (EBITDA – Capex) / (Interest + any required principal). Assume 5% mandatory amortization on the senior debt.
* Propose a covenant package (maximum total leverage, minimum interest coverage, minimum DSCR or fixed-charge coverage) with reasonable headroom. Explain how you would size the revolver and any excess cash flow sweep.
* Comment on recovery expectations and what would constitute a “tight” vs “sponsor-friendly” structure. Flag any red flags for credit committees.

Typical mid-market LBO/acquisition leverage has historically sat in the 4–6x range depending on the cycle and industry; coverage is often the binding constraint in higher-rate environments.

## 2. Proposed Covenant Package

| Covenant | Level | Notes / Headroom |
| --- | --- | --- |
| Maximum Total Leverage | 7.50x | Sized to the USD 600mm term loan (7.5x LTM EBITDA). Leaves a small cushion to the ~7.8x interest-coverage limit. |
| Minimum Interest Coverage | 2.00x | Matches the base-case floor used for debt capacity. Tested quarterly on LTM EBITDA / LTM cash interest. |
| Minimum DSCR (or FCCR) | 1.15x | Provides modest headroom under the ~1.09x Year-1 base DSCR at USD 600mm. |
| Revolver Commitment | USD 25mm | Undrawn at close. Sized for working-capital needs + modest extra capacity (not pure leverage headroom). |
| Excess Cash Flow Sweep | 50% / 25% / 0% | Annual sweep of Excess Cash Flow: • 50% if Total Leverage > 5.0x • 25% if 4.0x–5.0x • 0% if ≤ 4.0x Applied to term-loan prepayment. |

## Imports and Helper Functions

In [21]:
import math
import numpy_financial as npf

import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option('display.expand_frame_repr', False)

def calc_ebitda_growth(proj_1y_ebitda: float, ltm_ebitda: float) -> float:
    """Calculate the EBITDA growth rate."""
    return math.log(proj_1y_ebitda/ltm_ebitda)

def calc_debt(ebitda: float, leverage: float) -> float:
    """Calculate debt amount from EBITDA and leverage multiple."""
    return ebitda * leverage

def calc_dscr(y1: list) -> float:
    """Calculate Year-1 DSCR from period records."""
    total_ebitda = 0
    total_capex = 0
    total_interest = 0
    total_required_principal = 0
    
    for period in y1:
        total_ebitda += period['ebitda']
        total_capex += period['capex']
        total_interest = period['interest']
        total_required_principal += period['principal_payment']
        
    return (total_ebitda - total_capex)/(total_interest + total_required_principal)

def calc_interest(rate: float, debt: float, n_per: int) -> float:
    """Calculate period interest."""
    period_rate = rate / n_per
    interest = period_rate * debt
    return interest

def calc_principal_terms(debt: float, amortization: float, term_structure: int) -> int:
    """Calculate periodic principal payment and number of periods to maturity."""
    term_principal = debt * (amortization / term_structure)
    terms_to_mat = int(debt / term_principal)
    return (term_principal, terms_to_mat)

def grow_ebitda(ebitda: float, growth_rate: float, n_per: int):
    """Apply periodic EBITDA growth."""
    return ebitda + (ebitda * (growth_rate / n_per))

def get_year_from_df(df: pd.DataFrame, term_structure: int) -> dict:
    """Extract the first period(s) as records."""
    return df.iloc[:term_structure].to_dict(orient='records')

## Scenario Cash Flow Generator

In [20]:
def run_scenario(case: dict) -> pd.DataFrame:
    """
    Generate a multi-period cash flow schedule for a given case.
    Debt is sized off LTM EBITDA. Coverage and DSCR use projected Year-1 EBITDA.
    """
    lookup_tbl = {}
    #start_ebitda = case['ltm_ebitda']
    #grow_1y_ebitda = grow_ebitda(start_ebitda,case['growth_rate'],case['term_structure'])
        
    debt = calc_debt(case['ltm_ebitda'], case['leverage'])

    term_ebitda = case['proj_1y_ebitda'] / case['term_structure']
    
    term_principal, terms_to_mat = calc_principal_terms(debt, case['amortization'], case['term_structure'])
    
    for y in range(terms_to_mat):
        if debt == 0 or debt < 0:
            break
            
        year = y + 1
        
        # inside the loop
        term_interest = calc_interest(case['interest_rate'], debt, case['term_structure'])
        term_d_a = case['d_a'] / case['term_structure']
        tax_rate = case['tax_rate']
        term_dnwc_dt = case['dnwc_dy'] / case['term_structure']
        term_capex = case['capex'] / case['term_structure']
        
        interest_coverage = term_ebitda / term_interest if term_interest > 0 else float("inf")
        
        term_ebit = max(term_ebitda - term_d_a, 0)
        term_ebt  = max(term_ebit   - term_interest, 0)
        term_tax  = max(term_ebt    * tax_rate, 0)
        term_net_income = max(term_ebt - term_tax, 0)
        
        # Cash flow available for debt service (before principal)
        fcf = term_net_income + term_d_a - term_capex - term_dnwc_dt

        # Mandatory principal is paid if cash is available (or always, depending on how strict you want to be)
        principal_paid = min(term_principal, max(fcf, 0))   # optional: only pay what you can
        
        debt = debt - principal_paid if fcf > 0 else debt

        lookup_tbl[year] = {
            "year": year,
            "start_debt": debt,
            "ebitda": term_ebitda,
            "interest": term_interest,
            "interest_coverage": interest_coverage,
            "principal_payment": term_principal,
            "ebit": term_ebit,
            "ebt": term_ebt,
            "tax": term_tax,
            "net_income": term_net_income,
            "capex": term_capex,
            "fcf": fcf,
            "end_debt": debt,
        }

        term_ebitda = grow_ebitda(term_ebitda,case['growth_rate'],case['term_structure'])

    return pd.DataFrame.from_dict(lookup_tbl, orient="index")

## Leverage Optimizer

In [13]:
def maximize_leverage(case: dict, interest_coverage_floor: float) -> float: 
    """
    Estimate maximum total leverage (Debt/EBITDA) that keeps Year 1 interest coverage (EBITDA / cash interest) ≥ 2.0x in the base case
    and ≥1.5x in the downside case. Assume an all-in cash interest rate of 7.5% on senior debt.
    """

    increment = 0.01
    
    case['leverage'] = 1
    
    df = run_scenario(case)
    y1 = get_year_from_df(df, case['term_structure'])


    interest_coverage = 0
    
    for term in y1:
        interest_coverage += term['interest_coverage']

    if interest_coverage < interest_coverage_floor:
        raise ValueError("start case coverage cannot be less than coverage floor")

    stop = False

    while interest_coverage > interest_coverage_floor or interest_coverage == interest_coverage_floor:
        df = run_scenario(case)
        y1 = df.iloc[:case['term_structure']].to_dict(orient='records')
        for term in y1:
            interest_coverage = term['interest_coverage']
            if interest_coverage < interest_coverage_floor or interest_coverage == interest_coverage_floor:
                stop = True
                break
        if stop:
            break
        case['leverage'] += increment

    return case['leverage']     

## Base Case Analysis

In [18]:
def base_case():
    case = {
        'ltm_ebitda': 80,
        'd_a': 10,
        'capex': 8,
        'dnwc_dy': 0,
        'tax_rate': 0.25,
        'interest_rate': 0.075,
        'amortization': 0.05,
        'proj_1y_ebitda': 86,
        'term_structure': 1
    }

    case['growth_rate'] = math.log(case['proj_1y_ebitda'] / case['ltm_ebitda'])
    
    case['leverage'] = maximize_leverage(case, 2.0)

    print(f"Maximum leverage in the base case is {case['leverage']:.2f}x")

    #case['debt'] = calc_debt(case['ltm_ebitda'], case['leverage'] )

    df = run_scenario(case)

    y1 = get_year_from_df(df, case['term_structure'])
    
    dscr = calc_dscr(y1)

    print(f"Year 1 Base DSCR = {dscr:.2f}\n")
    print(f"Base Case First 10 Periods\n", df)

base_case()

Maximum leverage in the base case is 7.17x
Year 1 Base DSCR = 1.09

Base Case First 10 Periods
     year  start_debt  ebitda  interest  interest_coverage  principal_payment   ebit    ebt   tax  net_income  capex    fcf  end_debt
1      1      546.86   86.00     43.02               2.00              28.68  76.00  32.98  8.25       24.74   8.00  26.74    546.86
2      2      518.18   92.22     41.01               2.25              28.68  82.22  41.20 10.30       30.90   8.00  32.90    518.18
3      3      489.50   98.89     38.86               2.54              28.68  88.89  50.03 12.51       37.52   8.00  39.52    489.50
4      4      460.82  106.04     36.71               2.89              28.68  96.04  59.33 14.83       44.50   8.00  46.50    460.82
5      5      432.14  113.71     34.56               3.29              28.68 103.71  69.15 17.29       51.86   8.00  53.86    432.14
6      6      403.46  121.93     32.41               3.76              28.68 111.93  79.52 19.88       59.

**Interpretation (Base Case)**  
At approximately 7.17x leverage the Year-1 interest coverage is exactly 2.0x and DSCR is ~1.09x. This is the theoretical maximum under the stated constraints. Most credit committees would prefer additional headroom (e.g., sizing closer to 5.5–6.5x) so that a modest EBITDA miss does not immediately breach covenants.

## Downside Case Analysis

In [22]:
def downside_case():
    case = {
        'ltm_ebitda': 80,
        'proj_1y_ebitda': 70,
        'd_a': 10,
        'capex': 8,
        'dnwc_dy': 0,
        'tax_rate': 0.25,
        'interest_rate': 0.075,
        'amortization': 0.05,
        'term_structure': 1,
        'growth_rate': 0
    }

    
    case['leverage'] = maximize_leverage(case, 1.5)

    print(f"Maximum leverage in the downside case is {case['leverage']:.2f}x")
    
    df = run_scenario(case)

    y1 = get_year_from_df(df, case['term_structure'])
    
    dscr = calc_dscr(y1)

    print(f"Year 1 Downside DSCR = {dscr:.2f}\n")
    
    print(f"Downside Case First 10 periods\n", df)

downside_case()

Maximum leverage in the downside case is 7.78x
Year 1 Downside DSCR = 0.80

Downside Case First 10 periods
     year  start_debt  ebitda  interest  interest_coverage  principal_payment  ebit   ebt   tax  net_income  capex   fcf  end_debt
1      1      610.41   70.00     46.68               1.50              31.12 60.00 13.32  3.33        9.99   8.00 11.99    610.41
2      2      597.75   70.00     45.78               1.53              31.12 60.00 14.22  3.55       10.66   8.00 12.66    597.75
3      3      584.37   70.00     44.83               1.56              31.12 60.00 15.17  3.79       11.38   8.00 13.38    584.37
4      4      570.24   70.00     43.83               1.60              31.12 60.00 16.17  4.04       12.13   8.00 14.13    570.24
5      5      555.32   70.00     42.77               1.64              31.12 60.00 17.23  4.31       12.92   8.00 14.92    555.32
6      6      539.55   70.00     41.65               1.68              31.12 60.00 18.35  4.59       13.76   8.0

**Interpretation (Downside Case)**  
At approximately 7.78x leverage the Year-1 interest coverage is 1.5x and DSCR falls to ~0.80x. The flat \$70 million EBITDA assumption is a one-time haircut consistent with standard credit underwriting practice for an initial capacity test. Continuous negative growth was deliberately avoided so the Year-1 coverage and DSCR tests remain clear and comparable.

## 8. Recovery Expectations & Credit Committee Flags

**Recovery**  
A generic 40% recovery assumption is conservative for senior secured paper on a stable industrial asset. More realistic downside recovery for a first-lien facility is typically in the 70–80% range, further supported by the excess cash flow sweep that reduces the outstanding balance over time.

**Red Flags at Maximum Leverage**
- Thin interest-coverage cushion (exactly at the 2.0x base floor)
- Sub-1.20x DSCR in the base case (~1.09x)
- High opening leverage relative to historical mid-market norms (4–6x)
- Limited liquidity buffer if the revolver remains small

**Tight vs. Sponsor-Friendly**
- *Tight*: Opening leverage and covenants set at the model maximum, DSCR < 1.15x, modest revolver.
- *Sponsor-friendly*: Leverage covenant 0.5–0.75x higher, interest coverage covenant at 1.75x, larger undrawn revolver, lower ECF sweep percentages.

For a robust structure, sizing the term loan to ~5.0–5.5x LTM EBITDA would produce Year-1 interest coverage ≥ 2.8x and DSCR ≥ 1.40x in the base case.

## 9. Key Takeaways & Skills Demonstrated

- Maximum theoretical leverage under the stated coverage floors is ~7.17x (base) / ~7.78x (downside).
- DSCR with 5% amortization is the tighter constraint and leaves limited cushion.
- A practical credit structure would typically stop short of the absolute maximum to preserve headroom.
- Skills demonstrated: financial modeling, scenario/stress testing, interest coverage and DSCR calculations, covenant design, and identification of credit red flags using Python (pandas).

---
*This is a self-directed technical exercise. All figures are illustrative and based on the simplified assumptions stated above.*